In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt

root_file = "tumor.root"
tree_name = "t"

E0 = 662.0
me = 511.0

phantom_vlms = [2, 3]
detector_vlms = [4, 5, 6]

xmin, xmax = -5, 5
ymin, ymax = -5, 5
nx, ny = 120, 120

image = np.zeros((ny, nx))

xgrid = np.linspace(xmin, xmax, nx)
ygrid = np.linspace(ymin, ymax, ny)

f = ROOT.TFile.Open(root_file)
t = f.Get(tree_name)

for entry in t:

    scatter_points = []
    detector_hits = []

    n = min(len(entry.x), len(entry.y), len(entry.z),
            len(entry.vlm), len(entry.pdg), len(entry.et))

    for i in range(n):

        x = entry.x[i]
        y = entry.y[i]
        z = entry.z[i]
        vlm = entry.vlm[i]
        pdg = entry.pdg[i]
        et = entry.et[i]

        # Compton scatter inside phantom/tumor
        if vlm in phantom_vlms and pdg == 22:
            scatter_points.append((x, y, z))

        # Gamma reaching detector
        if vlm in detector_vlms and pdg == 22 and et > 0:
            detector_hits.append((x, y, z, et))

    for sx, sy, sz in scatter_points:
        for dx, dy, dz, Eprime in detector_hits:

            cos_theta = 1 - me * (1/Eprime - 1/E0)

            if cos_theta < -1 or cos_theta > 1:
                continue

            theta = np.arccos(cos_theta)

            axis = np.array([dx-sx, dy-sy, dz-sz])
            axis_norm = np.linalg.norm(axis)

            if axis_norm == 0:
                continue

            axis = axis / axis_norm

            for ix, X in enumerate(xgrid):
                for iy, Y in enumerate(ygrid):

                    point = np.array([X-sx, Y-sy, 0-sz])
                    r = np.linalg.norm(point)

                    if r == 0:
                        continue

                    point = point / r

                    angle = np.arccos(np.clip(np.dot(axis, point), -1, 1))

                    if abs(angle - theta) < 0.03:
                        image[iy, ix] += 1
                        plt.figure(figsize=(7,6))
plt.imshow(image, extent=[xmin, xmax, ymin, ymax],
           origin="lower", cmap="hot")
plt.colorbar(label="Back-projected counts")
plt.xlabel("x [cm]")
plt.ylabel("y [cm]")
plt.title("Compton Cone Back-Projection Image")
plt.show()

/tmp/ipykernel_30/3981779888.py:84: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(7,6))


<Figure size 700x600 with 0 Axes>